In [2]:
import os.path

import pandas as pd
import numpy as np
import rasterio

In [13]:
CSV_INVENTORY_PATH = "/home/martin/repositories/RemoteSensing/Download/ndvi_output/inventory.csv"

OUT_NO_NAN_DIR = "/home/martin/repositories/RemoteSensing/Download/ndvi_NAN_fill"
LOG_PATH_NO_NAN_np = "/home/martin/repositories/RemoteSensing/Download/ndvi_NAN_fill/inventory_np"
LOG_PATH_NO_NAN_tif = "/home/martin/repositories/RemoteSensing/Download/ndvi_NAN_fill/inventory_tif"

## Fill N/A in composites

In [8]:
composites_inventory = pd.read_csv(CSV_INVENTORY_PATH, index_col=0, header = 0)

min_year = composites_inventory.year.min()
max_year = composites_inventory.year.max()
months = [1,4,7,10]

composites_inventory = composites_inventory.set_index(["year", "month"])
composites_inventory.sort_index(inplace=True)

def fillNAN(orig_series, out_series, nan_ind):
    
    offset = 1
    
    while True:
        ind_r = nan_ind+[offset,0,0] if nan_ind[0]+offset < len(orig_series) else [len(orig_series)-1, nan_ind[1], nan_ind[2]]
        right = orig_series[ind_r[0], ind_r[1], ind_r[2]]
        ind_l = nan_ind-[offset,0,0] if nan_ind[0]-offset >= 0 else [0, nan_ind[1], nan_ind[2]]
        left = orig_series[ ind_l[0], ind_l[1], ind_l[2] ]
        
        if not np.isnan(left) and not np.isnan(right):
            out_series[nan_ind[0], nan_ind[1], nan_ind[2]] = (left+right)/2
            return
        
        if np.isnan(left) and np.isnan(right):
            offset+=1
            continue
        else:
            out_series[nan_ind[0], nan_ind[1], nan_ind[2]] = left if np.isnan(right) else right
            return


# load all
series = []
for year in range(min_year,max_year+1): 
    for month in months:
        row = composites_inventory.loc[(year,month)]
        
        tif = rasterio.open(row["path"])
        as_np = tif.read()
        tif.close()
        
        series.append(as_np)
        
series = np.array(series)
series = series.squeeze()
print(series.shape)

print("NAN BEFORE fill:\n", (np.isnan(series)).sum())
# fill
filled_series = np.copy(series)
for nan in  np.argwhere(np.isnan(series)):
    fillNAN(series, filled_series, nan)

print("NAN after fill:\n", np.argwhere(np.isnan(filled_series)))


(44, 2500, 2500)
NAN BEFORE fill:
 [[   0    0    0]
 [   0    0    1]
 [   0    0    2]
 ...
 [  43 2398 2254]
 [  43 2404 2240]
 [  43 2405 2240]]
NAN after fill:
 []


In [18]:
# save
row = composites_inventory.loc[(2015,10)]
with rasterio.open(row["path"]) as gtif:
    gtif_profile = gtif.profile


print(filled_series.shape)
inventory_log_np = pd.DataFrame(columns=[ "year", "month", "path"])
inventory_log_tif = pd.DataFrame(columns=[ "year", "month", "path"])
index = 0
for year in range(min_year,max_year+1): 
    for month in months:
        composite_path_np = os.path.join(OUT_NO_NAN_DIR, f"{year}_{month}.npy")
        np.save(composite_path_np, filled_series[index])
        
        inventory_log_np.loc[len(inventory_log_np)] = [year, month, composite_path_np]
        
        composite_path_tif = os.path.join(OUT_NO_NAN_DIR, f"{year}_{month}.tif")
        with rasterio.open(composite_path_tif, 'w', **gtif_profile) as gtif:
            gtif.write(filled_series[index],1)
        inventory_log_tif.loc[len(inventory_log_tif)] = [year, month, composite_path_tif]
        
        index+=1
        
inventory_log_np.to_csv(LOG_PATH_NO_NAN_np)
inventory_log_tif.to_csv(LOG_PATH_NO_NAN_tif)
print("done")

(44, 2500, 2500)
done
